# 5G Network Intelligence - Week 2 baselines
Run `python -m scripts.run_baselines --segment val` from the repo root first.
This notebook reads validation artifacts only. Do not use holdout for model selection.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
benchmark = pd.read_csv(root / 'reports/baseline_benchmark_val.csv')
pooled = benchmark[benchmark.load_tier == 'All']
display(pooled[['model', 'horizon', 'mae', 'rmse', 'wape', 'macro_mae', 'n_valid', 'n_excluded']])


In [ ]:
pooled.pivot(index='horizon', columns='model', values='mae').plot.bar()
plt.ylabel('Validation MAE - DLPRB usage')
plt.title('Fixed-model rolling-origin validation')
plt.tight_layout()
plt.show()
display(benchmark[['model', 'horizon', 'load_tier', 'active_only_mae', 'zero_fp_rate', 'n_active', 'n_zero']])


In [ ]:
display(Markdown((root / 'reports/baseline_analysis.md').read_text(encoding='utf-8')))
metadata = json.loads((root / 'reports/baselines_run_metadata.json').read_text())
display(metadata)


In [ ]:
# Optional replay inspection; the archive is generated locally, not tracked in Git.
with np.load(root / 'reports/forecasts/baselines_val.npz', allow_pickle=False) as archive:
    replay_meta = json.loads(str(archive['metadata']))
    display(replay_meta['records'])
    prefix = replay_meta['records'][0]['prefix']
    origins = archive[f'{prefix}_origins']
    actual = archive[f'{prefix}_actual'][:, 0]
    predicted = archive[f'{prefix}_predicted'][:, 0]
    plt.plot(origins, actual, label='Actual at target')
    plt.plot(origins, predicted, label='Prediction')
    plt.xlabel('Forecast origin index')
    plt.ylabel('DLPRB usage - first beam')
    plt.legend()
    plt.show()
